# 2048 with Local LLM (LM Studio)

This notebook compares the **untrained Qwen3-8B baseline** against the **trained Qwen3-8B-2048** model on how well they play 2048.

Both models run locally via [LM Studio](https://lmstudio.ai). Each model plays 10 games and the results are compared side-by-side.

### Prerequisites

1. Install [LM Studio](https://lmstudio.ai)
2. Download **Qwen3-8B** from the LM Studio model catalog (baseline)
3. Run `2048-cloud.ipynb` to train and download the GGUF model, or manually download from [HuggingFace](https://huggingface.co/nathanpua/qwen3-8b-2048-gguf)
4. Load **both models** in LM Studio
5. Start the local server: click the **Local Server** tab, load a model, then click **Start Server**
6. The server runs at `http://localhost:1234/v1` by default

**Note:** Both models support Qwen3's thinking mode. The `extract_move()` function handles it transparently whether thinking is enabled or not. However, we disable this for lower latency across games

### Installation

In [ ]:
%pip install openai -q

### Connect to LM Studio

Verify that LM Studio is running and **both models are loaded** (baseline `qwen/qwen3-8b` and trained `nathanpua/qwen3-8b-2048`).

In [ ]:
from openai import AsyncOpenAI

LM_STUDIO_URL = "http://localhost:1234/v1"

client = AsyncOpenAI(
    base_url=LM_STUDIO_URL,
    api_key="lm-studio",  # LM Studio doesn't require a real key
)

# List available models to verify connection
models = await client.models.list()
for m in models.data:
    print(f"Model: {m.id}")

### Agentic Environment

The 2048 game engine. The winning value is set to **64** (instead of 2048) to keep games short and make evaluation practical on local hardware.

In [ ]:
import random
import string
import xml.etree.ElementTree as ET
from typing import Literal, TypedDict

WINNING_VALUE = 64


class TwentyFortyEightGame(TypedDict):
    id: str
    board: list[list[int | None]]


def populate_random_cell(game: TwentyFortyEightGame) -> None:
    all_clear_coordinates = [
        (i, j)
        for i in range(len(game["board"]))
        for j in range(len(game["board"][i]))
        if game["board"][i][j] is None
    ]
    random_clear_coordinates = random.choice(all_clear_coordinates)
    game["board"][random_clear_coordinates[0]][random_clear_coordinates[1]] = (
        2 if random.random() < 0.9 else 4
    )


def generate_game(board_length: int = 4) -> TwentyFortyEightGame:
    id = "".join(random.choices(string.ascii_letters + string.digits, k=6))
    game = {
        "id": id,
        "board": [[None for _ in range(board_length)] for _ in range(board_length)],
    }
    populate_random_cell(game)
    populate_random_cell(game)
    return game


def render_board(game: TwentyFortyEightGame) -> str:
    board = game["board"]
    max_cell_width = max(
        [len(str(cell)) for row in board for cell in row if cell is not None]
    )
    board_str = ""
    for row in board:
        board_str += "|".join(
            [
                str(cell).rjust(max_cell_width)
                if cell is not None
                else "_".rjust(max_cell_width)
                for cell in row
            ]
        )
        board_str += "\n"
    return board_str


def condense_sequence(sequence: list[int | None]) -> list[int | None]:
    condensed_sequence = []
    gapless_sequence = [cell for cell in sequence if cell is not None]
    i = 0
    while i < len(gapless_sequence):
        if (
            i + 1 < len(gapless_sequence)
            and gapless_sequence[i] == gapless_sequence[i + 1]
        ):
            condensed_sequence.append(gapless_sequence[i] * 2)
            i += 2
        else:
            condensed_sequence.append(gapless_sequence[i])
            i += 1
    return condensed_sequence + [None] * (4 - len(condensed_sequence))


def condense_board(
    game: TwentyFortyEightGame, direction: Literal["left", "right", "up", "down"]
) -> None:
    if direction == "left":
        for row in game["board"]:
            condensed_row = condense_sequence(row)
            for i in range(len(row)):
                row[i] = condensed_row[i]
    if direction == "right":
        for row in game["board"]:
            reversed_row = row[::-1]
            condensed_row = condense_sequence(reversed_row)[::-1]
            for i in range(len(row)):
                row[i] = condensed_row[i]
    if direction == "up":
        for col_index in range(len(game["board"][0])):
            column = [row[col_index] for row in game["board"]]
            condensed_column = condense_sequence(column)
            for row_index in range(len(column)):
                game["board"][row_index][col_index] = condensed_column[row_index]
    if direction == "down":
        for col_index in range(len(game["board"][0])):
            column = [row[col_index] for row in game["board"]]
            reversed_column = column[::-1]
            condensed_column = condense_sequence(reversed_column)[::-1]
            for row_index in range(len(column)):
                game["board"][row_index][col_index] = condensed_column[row_index]


def has_empty_cell(game: TwentyFortyEightGame) -> bool:
    return any(cell is None for row in game["board"] for cell in row)


def apply_agent_move(game: TwentyFortyEightGame, move_xml: str) -> None:
    direction = None
    try:
        root = ET.fromstring(move_xml)
        direction = root.text
    except Exception:
        raise ValueError("Invalid xml")
    if direction not in ["left", "right", "up", "down"]:
        raise ValueError("Invalid direction")
    condense_board(game, direction)
    if has_empty_cell(game):
        populate_random_cell(game)


def max_cell_value(game: TwentyFortyEightGame) -> int:
    return max([cell for row in game["board"] for cell in row if cell is not None])


def check_game_finished(game: TwentyFortyEightGame) -> bool:
    if max_cell_value(game) >= WINNING_VALUE:
        return True
    if any(cell is None for row in game["board"] for cell in row):
        return False
    return True


def total_board_value(game: TwentyFortyEightGame) -> int:
    return sum([cell for row in game["board"] for cell in row if cell is not None])


print("Game environment ready.")

### Shared Helpers

System prompt, move extraction, and benchmark runner used by both evaluation sections.

In [ ]:
import re
import asyncio

SYSTEM_PROMPT = (
    "You are an excellent 2048 player. Always choose the move most likely to lead to "
    "combine cells to eventually reach the number 2048. Optional moves are 'left', "
    "'right', 'up', 'down'. Return your move as an XML object with a single property "
    "'move', like so: <move>left</move>"
)

# Resolve model IDs
available = await client.models.list()
model_ids = {m.id for m in available.data}

BASELINE_MODEL = "qwen/qwen3-8b"
TRAINED_MODEL = next((m for m in model_ids if "2048" in m), None)

print(f"Available models: {', '.join(sorted(model_ids))}")
print(f"\nBaseline: {BASELINE_MODEL} {'OK' if BASELINE_MODEL in model_ids else 'NOT FOUND'}")
print(f"Trained:  {TRAINED_MODEL} {'OK' if TRAINED_MODEL in model_ids else 'NOT FOUND'}")

if BASELINE_MODEL not in model_ids:
    raise RuntimeError(f"Baseline model {BASELINE_MODEL} not loaded in LM Studio")
if not TRAINED_MODEL:
    raise RuntimeError("Trained model (nathanpua/qwen3-8b-2048) not loaded in LM Studio")


def extract_move(response) -> str:
    """Extract the <move>...</move> from response content or reasoning.

    The trained model outputs just '<move>left</move>' (8 tokens).
    The untrained baseline may reason for 200+ tokens before the tag.
    extract_move handles both cases by scanning the full output for the tag.
    """
    msg = response.choices[0].message
    for text in [msg.content, getattr(msg, "reasoning_content", None)]:
        if text:
            match = re.search(r"<move>\s*(left|right|up|down)\s*</move>", text)
            if match:
                return f"<move>{match.group(1)}</move>"
    raise ValueError("No valid move found in response")


async def play_game(game_num: int, model_id: str) -> dict:
    game = generate_game()
    move_number = 0
    invalid = False

    while not check_game_finished(game):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": render_board(game)},
        ]

        try:
            response = await client.chat.completions.create(
                model=model_id,
                messages=messages,
                max_completion_tokens=512,
            )
            content = extract_move(response)
        except Exception as e:
            print(f"Game {game_num + 1}: API error on move {move_number}: {e}")
            break

        assert isinstance(content, str)

        try:
            apply_agent_move(game, content)
            move_number += 1
        except ValueError:
            invalid = True
            break

    max_value = max_cell_value(game)
    board_value = total_board_value(game)
    won = max_value >= WINNING_VALUE

    status = "WON" if won else ("INVALID" if invalid else "LOST")
    print(f"Game {game_num + 1}: {status} | moves: {move_number} | max: {max_value} | board: {board_value}")

    return {
        "game": game_num + 1,
        "won": won,
        "invalid": invalid,
        "moves": move_number,
        "max_value": max_value,
        "board_value": board_value,
    }


async def run_benchmark(model_id: str, num_games: int = 10) -> dict:
    """Run games sequentially against LM Studio (single inference slot).

    Unlike vLLM in training (which supports continuous batching with max_num_seqs=4),
    LM Studio's llama.cpp server processes one request at a time. Parallel requests
    just add queuing overhead with no throughput benefit.

    Note: LM Studio's grammar parameter is unreliable (not consistently enforced),
    so we give the model enough token budget to reason and extract the move via regex.
    The trained model outputs ~8 tokens/move; the baseline may use ~200 tokens/move.
    """
    print(f"Benchmarking: {model_id}")
    print(f"Games: {num_games} (sequential)\n")

    results = []
    for i in range(num_games):
        result = await play_game(i, model_id)
        results.append(result)

    wins = sum(1 for r in results if r["won"])
    invalids = sum(1 for r in results if r["invalid"])
    avg_moves = sum(r["moves"] for r in results) / len(results)
    avg_max = sum(r["max_value"] for r in results) / len(results)
    avg_board = sum(r["board_value"] for r in results) / len(results)

    print(f"\n--- Summary: {model_id} ---")
    print(f"Wins:   {wins}/{num_games} ({wins / num_games * 100:.0f}%)")
    print(f"Invalid moves: {invalids}/{num_games}")
    print(f"Avg moves:  {avg_moves:.1f}")
    print(f"Avg max tile: {avg_max:.1f}")
    print(f"Avg board value: {avg_board:.1f}")

    return {
        "model": model_id,
        "win_rate": wins / num_games * 100,
        "wins": wins,
        "total": num_games,
        "invalids": invalids,
        "avg_moves": avg_moves,
        "avg_max": avg_max,
        "avg_board": avg_board,
    }

### Part 1: Baseline (Untrained Qwen3-8B)

Run 10 games sequentially with the untrained base model to establish a performance baseline.

In [ ]:
baseline = await run_benchmark(BASELINE_MODEL)

### Part 2: Trained (Qwen3-8B-2048)

Run 10 games with the model trained via GRPO on 20 steps x 18 games.

In [ ]:
trained = await run_benchmark(TRAINED_MODEL)

### Comparison

Side-by-side comparison of baseline vs trained model performance across all metrics.

In [ ]:
metrics = [
    ("Win Rate (%)", "win_rate"),
    ("Wins", "wins"),
    ("Invalid Moves", "invalids"),
    ("Avg Moves", "avg_moves"),
    ("Avg Max Tile", "avg_max"),
    ("Avg Board Value", "avg_board"),
]

b_name = baseline["model"].split("/")[-1]
t_name = trained["model"].split("/")[-1]
print(f"{'Metric':<20} {b_name:>16} {t_name:>16} {'Delta':>10}")
print("-" * 65)

for label, key in metrics:
    b = baseline[key]
    t = trained[key]
    d = t - b
    sign = "+" if d > 0 else ""
    fmt = ".0f" if key in ("wins", "invalids") else ".1f"
    print(f"{label:<20} {b:>{16}{fmt}} {t:>{16}{fmt}} {sign}{d:>{9}{fmt}}")

improvement = trained["win_rate"] - baseline["win_rate"]
if improvement > 0:
    print(f"\nTraining improved win rate by +{improvement:.0f} percentage points.")
elif improvement < 0:
    print(f"\nWin rate decreased by {improvement:.0f} percentage points after training.")
else:
    print(f"\nNo change in win rate.")